# Canonical v1 — Colab A100 一条龙执行器（Addendum v1.1）

流程：GPU 门 → 挂载 Drive → clone/checkout → 装依赖 → 完整测试门 → Smoke（primary + repeat + validator）→ Canonical core。

使用规则：
- **按顺序逐格运行**，任何一格失败就停下排查，不要跳过。
- 结果全部写入 Google Drive，会话断开后重开本 notebook、重跑到 canonical 那一格（**去掉 `--fresh`**）即可续跑。
- 先跑满前 3 个 seeds（42/123/2024）即可暂停做统计；规则见 `docs/paper_rebuild/PROTOCOL_ADDENDUM_V1_1.md` §1.5。

In [1]:
# [1] GPU 门：canonical core 必须是 A100（Colab 菜单: 代码执行程序 -> 更改运行时类型 -> A100 GPU）
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(out.stdout or out.stderr)
assert "A100" in out.stdout, "当前运行时不是 A100；请更改运行时类型后重跑本格"


NVIDIA A100-SXM4-40GB, 40960 MiB



In [2]:
# [2] 挂载 Google Drive 并创建输出根目录（所有结果持久化在这里）
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/diff_lora')
SMOKE_PRIMARY = BASE / 'stage2_smoke' / 'colab_primary'
SMOKE_REPEAT  = BASE / 'stage2_smoke' / 'colab_repeat'
CANONICAL     = BASE / 'canonical_v1'
for p in (SMOKE_PRIMARY.parent, CANONICAL):
    p.mkdir(parents=True, exist_ok=True)
print('输出根目录:', BASE)


Mounted at /content/drive
输出根目录: /content/drive/MyDrive/diff_lora


In [10]:
# [3] Clone 仓库并 checkout（COMMIT 留空 = 分支最新；填 40 位 hash 可精确固定）
COMMIT = ""  # 例如 "0123abcd..."；正式 canonical 建议填上并保持整轮一致

import subprocess, os
BRANCH = "codex/stage1-canonical-infrastructure"
REPO_URL = "https://github.com/LeafTraces/Diff-LoRA"
if not os.path.exists('/content/Diff-LoRA'):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, "/content/Diff-LoRA"], check=True)
os.chdir('/content/Diff-LoRA')
if COMMIT:
    subprocess.run(["git", "checkout", COMMIT], check=True)
head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True, check=True).stdout
assert not status, f"工作树不干净:\n{status}"
print("执行 commit:", head)


执行 commit: 3af3060e546d9004c5bf4a099da3221e1cf03d84


In [4]:
# [4] 安装依赖（Colab 自带 CUDA 版 torch，不要重装 torch）
%pip install -q transformers datasets accelerate scikit-learn tqdm
import torch, transformers, datasets, numpy
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| gpu", torch.cuda.get_device_name(0))
print("transformers", transformers.__version__, "| datasets", datasets.__version__, "| numpy", numpy.__version__)


torch 2.11.0+cu128 | cuda 12.8 | gpu NVIDIA A100-SXM4-40GB
transformers 5.13.1 | datasets 4.0.0 | numpy 2.0.2


In [5]:
import subprocess, re
p = subprocess.run(
    ["python", "-m", "unittest", "discover", "-s", "tests", "-p", "test_*.py"],
    cwd="/content/Diff-LoRA", capture_output=True, text=True)
out = p.stdout + p.stderr
print("\n".join(out.strip().splitlines()[-3:]))   # 总结行
for block in re.split(r"={70}", out):             # 每个失败的完整堆栈
    if block.strip().startswith(("ERROR:", "FAIL:")):
        print("=" * 70)
        print(block.strip()[:4000])

Ran 156 tests in 14.172s

FAILED (errors=1)
ERROR: test_stage2_freeze (unittest.loader._FailedTest.test_stage2_freeze)
----------------------------------------------------------------------
ImportError: Failed to import test module: test_stage2_freeze
Traceback (most recent call last):
  File "/usr/lib/python3.12/unittest/loader.py", line 396, in _find_test_path
    module = self._get_module_from_name(name)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/unittest/loader.py", line 339, in _get_module_from_name
    __import__(name)
  File "/content/Diff-LoRA/tests/test_stage2_freeze.py", line 31, in <module>
    from tests.test_stage2_validation import (
ModuleNotFoundError: No module named 'tests.test_stage2_validation'


----------------------------------------------------------------------
Ran 156 tests in 14.172s

FAILED (errors=1)


In [5]:
# [5] 完整测试门：必须 OK 才能继续（约 1–3 分钟）
!cd /content/Diff-LoRA && python -m unittest discover -s tests -p "test_*.py" 2>&1 | tail -5



FAILED (errors=1)
pass
fail
pass


In [ ]:
# [6] Smoke primary（tiny-data；含 manifest 初始化与官方 HANS 11 字段 digest 校验）
!cd /content/Diff-LoRA && python run_stage2_smoke.py \
  --mode primary --environment colab_a100 \
  --protocol docs/paper_rebuild/FROZEN_EXPERIMENT_PROTOCOL.md \
  --output-dir "{SMOKE_PRIMARY}" --fresh


In [ ]:
# [7] Smoke repeat（同一 runtime 重复 full_sr，用于 <=0.5pp 复现比较）
!cd /content/Diff-LoRA && python run_stage2_smoke.py \
  --mode repeat_full_sr --environment colab_a100 \
  --protocol docs/paper_rebuild/FROZEN_EXPERIMENT_PROTOCOL.md \
  --output-dir "{SMOKE_REPEAT}" --fresh


In [ ]:
# [8] Smoke validator：全部通过才允许进入 canonical core
!cd /content/Diff-LoRA && python validate_stage2_smoke.py \
  --root "{SMOKE_PRIMARY}" \
  --conditions standard_lora full_sr class_prior_reweight \
  --compare-repeat "{SMOKE_REPEAT}" \
  --canonical-dir "{CANONICAL}"


In [ ]:
# [9] 环境快照存档
!mkdir -p "{BASE}/manifests" && pip freeze > "{BASE}/manifests/pip_freeze_$(date +%Y%m%dT%H%M%S).txt" && nvidia-smi > "{BASE}/manifests/nvidia_smi_$(date +%Y%m%dT%H%M%S).txt" && echo saved


## Canonical core

- **首次**启动带 `--fresh`（要求 `canonical_v1` 目录为空）；之后每次续跑都**去掉 `--fresh`**。
- 执行顺序为 seed-major：seed_42 全部 → seed_123 全部 → seed_2024 全部 → …。每个 seed 约 3 小时（粗估）。
- 一个 Colab 会话跑不完很正常：断开后重开 notebook，跑格 [1]–[4]，然后直接跑下面这格（去掉 `--fresh`），已完成且校验通过的 run 会被自动跳过。
- **先 3 seeds**：`seed_2024` 目录下 6 个条件全部 `status.json = success` 后即可暂停，进入统计分析（Stage 4）。

In [ ]:
# [10] Canonical core（首次带 --fresh；续跑请删掉 --fresh 再运行）
!cd /content/Diff-LoRA && python run_canonical.py \
  --stage core \
  --protocol docs/paper_rebuild/FROZEN_EXPERIMENT_PROTOCOL.md \
  --output-dir "{CANONICAL}" --fresh


In [ ]:
# [11] 进度速览：各 seed/条件的 status
import json
from pathlib import Path
for status in sorted(Path(f"{CANONICAL}").glob("seed_*/*/status.json")):
    state = json.loads(status.read_text())
    print(f"{status.parent.parent.name}/{status.parent.name}: {state.get('state')}")
